# NPC-Sim v5 — Dual-LLM Pipeline Fine-Tuning (v1.6.0)
**Kaggle GPU: T4 x2 / P100 (16 GB VRAM)**

### Architecture v1.6.0 — Two-Model Pipeline
| Component | Base Model | Role | Output |
|-----------|-----------|------|--------|
| **Reasoner** | Hermes-3-Llama-3.2-3B + LoRA r=16 | Turkish CoT reasoning | 3-5 sentence internal monologue |
| **Formatter** | Llama 3.2 1B-**Instruct** + LoRA r=8 | CoT prose → strict JSON | `{reasoning, selected_action, emotion}` |

### v1.6.0 Training Fixes
- `packing=False` + `DataCollatorForCompletionOnlyLM`: loss only on assistant tokens, EOS always terminal
- Formatter base = **Instruct** variant (base model doesn't know chat-format tokens)
- Dataset generated locally with `gemma4:e4b` bootstrap CoT, uploaded as `rpg-dataset-v5`
- Formatter schema: `npc_id` removed (injected at runtime by `DualLLMBackend`)
- Multi-factor decision model labels (self_power vs perceived_threat + duty_pull)

### Pipeline
1. Load v5 pre-generated datasets from Kaggle input
2. **Phase A:** Fine-tune Hermes-3-Llama-3.2-3B on Turkish CoT reasoning (10k examples)
3. Trait stress test: 50 prompts × 5 archetypes → trait coherence ≥ 80%
4. **Phase B:** Fine-tune Llama 3.2 1B-Instruct on CoT→JSON formatting (~12k examples)
5. Full chain stress test + evaluate both models
6. Export LoRA adapters

## 0. Install Dependencies
Restart kernel & clear outputs after running this cell.

In [1]:
!pip install -q \
    transformers==4.46.3 \
    datasets==3.1.0 \
    peft==0.13.2 \
    trl==0.12.1 \
    bitsandbytes==0.44.1 \
    accelerate==1.1.1 \
    sentencepiece \
    scipy

!pip install -q -U bitsandbytes==0.43.3
!pip install -q triton==2.3.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.9/310.9 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.4/122.4 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 72.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behav

## 1. Load Pre-generated Datasets (v5)
Data was generated locally using `gemma4:e4b` bootstrap CoT via `Stateful_NPC/generator/npc_sim_generator_v2.py`
and uploaded to Kaggle as dataset `rpg-dataset-v5`.

Files:
- `train_reasoner.jsonl` — 10k CoT examples for the Reasoner (persona card + NPC state → Turkish monologue)
- `test_reasoner.jsonl` — 2k eval set
- `train_formatter.jsonl` — ~12k CoT→JSON pairs for the Formatter (17.5% paraphrase-augmented)

In [ ]:
import json, pathlib, os

DATASET_DIR = pathlib.Path("/kaggle/input/rpg-dataset-v5")

def _load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_reasoner  = _load_jsonl(DATASET_DIR / "train_reasoner.jsonl")
test_reasoner   = _load_jsonl(DATASET_DIR / "test_reasoner.jsonl")
train_formatter = _load_jsonl(DATASET_DIR / "train_formatter.jsonl")

print(f"Reasoner  train: {len(train_reasoner):,}   test: {len(test_reasoner):,}")
print(f"Formatter train: {len(train_formatter):,}")

## 2. Preview & Validate Datasets

In [ ]:
## 2. Preview & Validate Datasets

# --- Reasoner example ---
print("=" * 60)
print("REASONER TRAINING EXAMPLE (first 600 chars):")
print("=" * 60)
print(train_reasoner[0]["text"][:600])
print("...\n")

# --- Formatter example ---
print("=" * 60)
print("FORMATTER TRAINING EXAMPLE (first 600 chars):")
print("=" * 60)
print(train_formatter[0]["text"][:600])
print("...\n")

# --- Size report ---
print(f"  train_reasoner   {len(train_reasoner):>7,} examples")
print(f"  test_reasoner    {len(test_reasoner):>7,} examples")
print(f"  train_formatter  {len(train_formatter):>7,} examples")

---
# Phase A: Reasoner (Component A) — Llama 3.2 3B
Fine-tune on Turkish Chain-of-Thought reasoning.

Input: full NPC state JSON → Output: 3-5 sentence Turkish internal monologue.

The model learns **why** an NPC should act, ignoring JSON formatting entirely.

## A.1 Load Reasoner Dataset

In [ ]:
from datasets import Dataset, DatasetDict

ds_reasoner = DatasetDict({
    "train": Dataset.from_list(train_reasoner),
    "test":  Dataset.from_list(test_reasoner),
})

print(ds_reasoner)
print(f"\nReasonerTrain[0] snippet:")
print(ds_reasoner["train"][0]["text"][:300])

## A.2 Load Llama 3.2 3B + LoRA

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

REASONER_MODEL_ID = "NousResearch/Hermes-3-Llama-3.2-3B"

tokenizer_a = AutoTokenizer.from_pretrained(REASONER_MODEL_ID)
tokenizer_a.pad_token = tokenizer_a.eos_token
tokenizer_a.padding_side = "right"

model_a = AutoModelForCausalLM.from_pretrained(
    REASONER_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

print(f"Base model loaded: {REASONER_MODEL_ID}")
print(f"Memory: {model_a.get_memory_footprint() / 1e9:.2f} GB")

# --- LoRA ---
lora_config_a = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model_a = get_peft_model(model_a, lora_config_a)
model_a.enable_input_require_grads()
model_a.gradient_checkpointing_enable()

model_a.print_trainable_parameters()

2026-04-21 18:59:10.376449: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776797950.631498      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776797950.706204      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776797951.316425      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776797951.316494      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776797951.316497      23 computation_placer.cc:177] computation placer alr

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/955 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

Base model loaded: NousResearch/Hermes-3-Llama-3.2-3B
Memory: 6.43 GB
trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


## A.3 Train Reasoner

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

os.environ["TOKENIZERS_PARALLELISM"] = "false"

OUTPUT_DIR_A = "/kaggle/working/reasoner-lora"

training_args_a = TrainingArguments(
    output_dir=OUTPUT_DIR_A,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    dataloader_num_workers=2,
    optim="adamw_torch",
    max_grad_norm=0.3,
    group_by_length=True,
    gradient_checkpointing=True,
)

MAX_SEQ_LEN_A = 1024

# packing=False: each sequence is exactly one example — EOS is always at the end.
# DataCollatorForCompletionOnlyLM: loss computed only on assistant tokens (after the
# response_template header), so the model never learns to regenerate the system prompt.
response_template_a = "<|start_header_id|>assistant<|end_header_id|>\n\n"
collator_a = DataCollatorForCompletionOnlyLM(response_template_a, tokenizer=tokenizer_a)

trainer_a = SFTTrainer(
    model=model_a,
    tokenizer=tokenizer_a,
    train_dataset=ds_reasoner["train"],
    eval_dataset=ds_reasoner["test"],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN_A,
    packing=False,
    data_collator=collator_a,
    args=training_args_a,
)

print("[Phase A] SFTTrainer ready — packing=False, DataCollatorForCompletionOnlyLM active.")
print("Starting Reasoner training...")
trainer_a.train()

## A.3.1 Reasoner Trait Stress Test
50 hardcoded prompts × 5 archetypes — verifies the Reasoner produces trait-coherent Turkish CoT.

| Archetype | Condition | Expected CoT signal |
|-----------|-----------|---------------------|
| Brave | threat ≥ 0.78 | words: cesur / saldır / ileri |
| Coward | threat ≥ 0.50 | words: kaç / geri / kork |
| Pacifist | any threat | must NOT contain "saldır" |
| Devout | stress ≥ 0.60, no threat | words: dua / tanrı / ibadet |
| Greedy | gold in inv, no threat | words: ticaret / satış / altın |

Target: **trait coherence ≥ 80%**

In [ ]:
import json, re, torch

_VALID_ACTIONS = ["eat","drink","sleep","flee","gather","heal",
                  "attack","socialize","trade","work","pray","walk_to"]

_TEMPLATE = (
    "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
    "{system}<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"
    "{user}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
)

_SYS_R = (
    "Sen bir ortaçağ simülasyonundaki NPC'nin iç ses motorusun.\n"
    "Sana NPC'nin anlık durumunu JSON olarak göndereceğim.\n"
    "Görevin: NPC'nin ne yapması gerektiğini ve NEDEN yapması gerektiğini\n"
    "3-5 cümlelik Türkçe iç monolog olarak yaz.\n"
    "ASLA JSON üretme. ASLA action_id adı kullanma. Sadece düşün.\n"
    "Kişilik özellikleri, hayatta kalma ihtiyaçları, tehdit algısı,\n"
    "duygusal durum, hafıza ve sosyal bağlamı değerlendir."
)

def _st_state(arch, traits, fear, threat, stress, inv=None, occ="Guard"):
    return json.dumps({
        "arch": arch, "occ": occ, "faction": "CityWatch",
        "b5": {"e":0.7,"a":0.5,"c":0.7,"n":0.8 if fear > 0.4 else 0.2,"o":0.4},
        "traits": traits,
        "vitals": {"hp":85.0,"hp_max":100.0,"en":0.7,"hun":0.3,"thi":0.3,"str":stress},
        "emo": {"hap":0.4,"fear":fear,"ang":0.2,"mood":"Calm"},
        "inv": inv or [],
        "time": {"day":2,"hr":14.0},
        "pos": {"x":50.0,"z":50.0,"zone":"MarketSquare","landmark":"CentralFountain"},
        "percepts": [{"id":"wolf_01","tag":"Threat","sal":threat,"threat":threat}] if threat > 0 else [],
        "memories": [],
        "beliefs": [],
        "factions": {"CityWatch":0.5},
        "goals_top": "Patrol",
        "interrupt": threat >= 0.8,
        "valid_actions": _VALID_ACTIONS,
    }, ensure_ascii=False)

_PERSONAS = {
    "Brave":    "Mehmet, Şehir Muhafızları bünyesinde görev yapan bir muhafız. Cesur ve sadık biri olarak tanınır; dışadönüklük yüksek, nevrotiklik düşük.",
    "Coward":   "Fatma, kervansarayda çalışan bir hizmetli. Korkak biri olarak tanınır; nevrotiklik yüksek, dışadönüklük düşük.",
    "Pacifist": "İbrahim, köy kilisesinde görev yapan bir rahip. Barışsever biri olarak tanınır; uyumluluk çok yüksek, saldırganlık çok düşük.",
    "Devout":   "Ayşe, tapınakta dua eden bir din görevlisi. Dindar ve vicdanlı biri olarak tanınır; öz-denetim yüksek, bilinçlilik yüksek.",
    "Greedy":   "Mustafa, pazar alanında mal satan bir tüccar. Açgözlü biri olarak tanınır; uyumluluk düşük, dışadönüklük orta.",
}

# 50 prompts: 10 per archetype
# expected = target action keyword or "!action" (must NOT produce that action)
STRESS_PROMPTS = []

for t in [0.80,0.85,0.88,0.90,0.92,0.82,0.87,0.91,0.79,0.84]:
    STRESS_PROMPTS.append(("Brave",    _st_state("Brave",["Brave","Loyal"],   fear=0.10,threat=t,  stress=0.15), "attack"))
for t in [0.50,0.60,0.70,0.75,0.55,0.65,0.80,0.52,0.68,0.72]:
    STRESS_PROMPTS.append(("Coward",   _st_state("Coward",["Coward"],         fear=0.75,threat=t,  stress=0.40), "flee"))
for t in [0.80,0.85,0.90,0.60,0.70,0.75,0.88,0.65,0.82,0.77]:
    STRESS_PROMPTS.append(("Pacifist", _st_state("Pacifist",["Pacifist"],     fear=0.30,threat=t,  stress=0.20,occ="Priest"), "!attack"))
for s in [0.60,0.65,0.70,0.75,0.80,0.62,0.68,0.73,0.78,0.82]:
    STRESS_PROMPTS.append(("Devout",   _st_state("Devout",["Devout"],         fear=0.10,threat=0.0,stress=s,   occ="Priest"), "pray"))
for n in [5,10,3,7,2,8,4,6,9,1]:
    STRESS_PROMPTS.append(("Greedy",   _st_state("Greedy",["Greedy"],         fear=0.10,threat=0.0,stress=0.10,
                                                  inv=[{"id":"gold","n":n},{"id":"food","n":1}],occ="Merchant"), "trade"))

# Keyword heuristics per archetype (checked in generated CoT)
_KEYWORDS = {
    "Brave":    (["cesur","saldır","ileri","kork değil","tehdit","güç"], []),
    "Coward":   (["kaç","geri","kork","tehlike","güvenli"], []),
    "Pacifist": ([], ["saldır"]),         # must NOT contain
    "Devout":   (["dua","tanrı","ibadet","kutsal","ruh"], []),
    "Greedy":   (["ticaret","satış","altın","kar","fırsat","zengin"], []),
}

# Run inference
model_a.eval()
DEVICE = next(model_a.parameters()).device
passed = 0
results = {k: {"pass":0,"fail":0} for k in _PERSONAS}

for arch, state_json, expected in STRESS_PROMPTS:
    persona = _PERSONAS[arch]
    user_turn = f"{persona}\n\n{state_json}"
    prompt = _TEMPLATE.format(system=_SYS_R, user=user_turn)

    enc = tokenizer_a(prompt, return_tensors="pt", truncation=True, max_length=1024).to(DEVICE)
    with torch.no_grad():
        out = model_a.generate(**enc, max_new_tokens=300, do_sample=False,
                               pad_token_id=tokenizer_a.eos_token_id)
    cot = tokenizer_a.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    cot = cot.split("<|eot_id|>")[0].strip().lower()

    pos_kw, neg_kw = _KEYWORDS[arch]
    has_pos = any(k in cot for k in pos_kw) if pos_kw else True
    has_neg = any(k in cot for k in neg_kw)
    fmt_ok = (not cot.startswith("{")) and (50 <= len(cot) <= 650)

    ok = fmt_ok and has_pos and (not has_neg)
    if ok:
        passed += 1
        results[arch]["pass"] += 1
    else:
        results[arch]["fail"] += 1

coherence = passed / len(STRESS_PROMPTS)
print(f"\nReasoner Trait Stress Test — {len(STRESS_PROMPTS)} prompts")
print(f"{'Archetype':<12} {'Pass':>4} {'Fail':>4}")
print("-" * 24)
for arch, r in results.items():
    print(f"  {arch:<10} {r['pass']:>4}  {r['fail']:>4}")
print("-" * 24)
print(f"  Trait coherence: {coherence:.1%}  (target ≥ 80%)")
if coherence < 0.80:
    print("  WARNING: below 80% threshold — check training data quality")

## A.4 Save Reasoner Adapter

In [7]:
ADAPTER_DIR_A = "/kaggle/working/reasoner-lora-final"

trainer_a.model.save_pretrained(ADAPTER_DIR_A)
tokenizer_a.save_pretrained(ADAPTER_DIR_A)

print(f"Reasoner adapter saved to: {ADAPTER_DIR_A}")
for f in os.listdir(ADAPTER_DIR_A):
    size = os.path.getsize(os.path.join(ADAPTER_DIR_A, f)) / 1e6
    print(f"  {f}: {size:.1f} MB")

Reasoner adapter saved to: /kaggle/working/reasoner-lora-final
  README.md: 0.0 MB
  tokenizer.json: 17.2 MB
  adapter_model.safetensors: 97.3 MB
  tokenizer_config.json: 0.1 MB
  special_tokens_map.json: 0.0 MB
  adapter_config.json: 0.0 MB


## A.5 Evaluate Reasoner
Component A should output Turkish internal monologue (NOT JSON).

Checks:
- Output length is 50-500 chars
- Output does NOT start with `{` (that would mean it's producing JSON instead of reasoning)
- Output contains Turkish language markers

In [8]:
import json
import torch

BATCH_SIZE = 8
MAX_NEW_TOK = 300
DEVICE = next(model_a.parameters()).device

eval_samples_a = ds_reasoner["test"].select(range(min(100, len(ds_reasoner["test"]))))

ass_tag = "<|start_header_id|>assistant<|end_header_id|>\n\n"

prompts_a = []
for sample in eval_samples_a:
    text = sample["text"]
    p_end = text.find(ass_tag)
    if p_end == -1:
        continue
    prompts_a.append(text[:p_end + len(ass_tag)])

print(f"Evaluating {len(prompts_a)} Reasoner examples...")

good_length = 0
no_json = 0
model_a.eval()

for batch_start in range(0, len(prompts_a), BATCH_SIZE):
    batch = prompts_a[batch_start : batch_start + BATCH_SIZE]

    tokenizer_a.padding_side = "left"
    enc = tokenizer_a(
        batch, return_tensors="pt", padding=True,
        truncation=True, max_length=1024,
    ).to(DEVICE)

    with torch.no_grad():
        out_ids = model_a.generate(
            **enc, max_new_tokens=MAX_NEW_TOK,
            do_sample=False, pad_token_id=tokenizer_a.eos_token_id,
            eos_token_id=tokenizer_a.eos_token_id,
        )

    new_ids = out_ids[:, enc["input_ids"].shape[1]:]
    decoded = tokenizer_a.batch_decode(new_ids, skip_special_tokens=True)

    for gen in decoded:
        gen = gen.split("<|eot_id|>")[0].strip()
        if 50 <= len(gen) <= 600:
            good_length += 1
        if not gen.strip().startswith("{"):
            no_json += 1

    done = min(batch_start + BATCH_SIZE, len(prompts_a))
    print(f"  [{done}/{len(prompts_a)}] good_len={good_length}  no_json={no_json}")

tokenizer_a.padding_side = "right"
n = len(prompts_a)
print(f"\n{'='*50}")
print(f"Reasoner Eval -- {n} examples:")
print(f"  Good length (50-600 chars): {good_length/n*100:.1f}%  (target >=90%)")
print(f"  Non-JSON output (correct):  {no_json/n*100:.1f}%  (target >=95%)")
print(f"{'='*50}")

Evaluating 100 Reasoner examples...


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


  [8/100] good_len=8  no_json=8
  [16/100] good_len=16  no_json=16
  [24/100] good_len=24  no_json=24
  [32/100] good_len=32  no_json=32
  [40/100] good_len=40  no_json=40
  [48/100] good_len=48  no_json=48
  [56/100] good_len=56  no_json=56
  [64/100] good_len=64  no_json=64
  [72/100] good_len=71  no_json=72
  [80/100] good_len=79  no_json=80
  [88/100] good_len=87  no_json=88
  [96/100] good_len=95  no_json=96
  [100/100] good_len=99  no_json=100

Reasoner Eval -- 100 examples:
  Good length (50-600 chars): 99.0%  (target >=90%)
  Non-JSON output (correct):  100.0%  (target >=95%)


## A.6 Unload Reasoner (Free VRAM for Phase B)

In [9]:
import gc

del model_a, trainer_a
gc.collect()
torch.cuda.empty_cache()

print(f"VRAM freed. Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print("Ready for Phase B.")

VRAM freed. Allocated: 0.02 GB
Ready for Phase B.


---
# Phase B: Formatter (Component B) — Llama 3.2 1B
Fine-tune on Turkish NL → strict JSON extraction.

Input: Component A's Turkish reasoning text → Output: strictly typed JSON response.

The model learns **schema compliance**, not reasoning. It's a "dumb but obedient" translator.

## B.1 Load Formatter Dataset

In [ ]:
from datasets import Dataset

ds_formatter = Dataset.from_list(train_formatter).train_test_split(test_size=0.1, seed=42)

print(ds_formatter)
print(f"\nFormatterTrain[0] snippet:")
print(ds_formatter["train"][0]["text"][:400])

## B.2 Load Llama 3.2 1B + LoRA

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

# Must be Instruct variant — base model doesn't know chat-format tokens (<|start_header_id|> etc.)
FORMATTER_MODEL_ID = "NousResearch/Llama-3.2-1B-Instruct"

tokenizer_b = AutoTokenizer.from_pretrained(FORMATTER_MODEL_ID)
tokenizer_b.pad_token = tokenizer_b.eos_token
tokenizer_b.padding_side = "right"

model_b = AutoModelForCausalLM.from_pretrained(
    FORMATTER_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

print(f"Base model loaded: {FORMATTER_MODEL_ID}")
print(f"Memory: {model_b.get_memory_footprint() / 1e9:.2f} GB")

lora_config_b = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model_b = get_peft_model(model_b, lora_config_b)
model_b.enable_input_require_grads()
model_b.gradient_checkpointing_enable()

model_b.print_trainable_parameters()

## B.3 Train Formatter

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

OUTPUT_DIR_B = "/kaggle/working/formatter-lora"

training_args_b = TrainingArguments(
    output_dir=OUTPUT_DIR_B,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    dataloader_num_workers=2,
    optim="adamw_torch",
    max_grad_norm=0.3,
    group_by_length=True,
    gradient_checkpointing=True,
)

MAX_SEQ_LEN_B = 512

# Same packing=False + collator fix as Reasoner — Formatter context is small but same invariant applies.
response_template_b = "<|start_header_id|>assistant<|end_header_id|>\n\n"
collator_b = DataCollatorForCompletionOnlyLM(response_template_b, tokenizer=tokenizer_b)

trainer_b = SFTTrainer(
    model=model_b,
    tokenizer=tokenizer_b,
    train_dataset=ds_formatter["train"],
    eval_dataset=ds_formatter["test"],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN_B,
    packing=False,
    data_collator=collator_b,
    args=training_args_b,
)

print("[Phase B] SFTTrainer ready — packing=False, DataCollatorForCompletionOnlyLM active.")
print("Starting Formatter training...")
trainer_b.train()

## B.3.1 Formatter Trait Stress Test
50 pre-written Turkish CoT texts (10 per archetype) fed directly to the Formatter.
Checks JSON parse rate, valid `action_id`, and that `npc_id` is absent from output.

| Archetype | CoT intent | Expected action_id |
|-----------|-----------|---------------------|
| Brave (×10) | saldırmalıyım / ileri atılmak | `attack` |
| Coward (×10) | kaçmam / geri çekilmek | `flee` |
| Pacifist (×10) | barışçıl / şiddet yok | `flee` or `pray` |
| Devout (×10) | dua / Tanrıya sığınmak | `pray` |
| Greedy (×10) | ticaret / altın satmak | `trade` |

Targets: **JSON parse ≥ 95%**, **valid action_id ≥ 90%**, **npc_id absent = 100%**

In [ ]:
import json, torch

_VALID_ACTIONS_B = ["eat","drink","sleep","flee","gather","heal",
                    "attack","socialize","trade","work","pray","walk_to"]

_SYS_F = (
    "Sen bir NPC simülasyonu için JSON dönüştürücüsün.\n"
    "Sana bir NPC'nin ne yapmak istediğini Türkçe olarak anlatacağım.\n"
    "Bunu aşağıdaki JSON şemasına dönüştür:\n"
    '{"reasoning":"<girdiyi kopyala>",'
    '"selected_action":{"action_id":"<listeden>","target_id":null,"dialogue":null},'
    '"emotion":"<tek kelime>"}\n'
    'valid_actions: ["eat","drink","sleep","flee","gather","heal",'
    '"attack","socialize","trade","work","pray","walk_to"]\n'
    "SADECE JSON yaz. Kod bloğu veya açıklama ekleme."
)

_TEMPLATE_B = (
    "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
    "{system}<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"
    "{user}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
)

# Pre-written CoT strings — feed to Formatter, check JSON output + action_id
FORMATTER_STRESS_INPUTS = [
    # Brave × 10 → attack
    ("Tehdit çok yakın, cesaretim beni öne itiyor. Saldırmalıyım, geri adım atmak utanç olur.", "attack"),
    ("Korkmak yok. Güçlüyüm ve göreve bağlıyım. Bu tehdidi ortadan kaldırmam gerekiyor.", "attack"),
    ("Rakibim güçlü ama ben daha güçlüyüm. İleri atılmak tek seçenek.", "attack"),
    ("Sadakatim ve cesaretim beni bu savaşa çekiyor. Saldırıya geçiyorum.", "attack"),
    ("Tehdit büyük ama benim gücüm daha büyük. Korkuyu bir kenara bırakıp saldırıyorum.", "attack"),
    ("Görevim beni korumayı emreder. Saldırarak tehdidi bertaraf etmeliyim.", "attack"),
    ("Bu bölgeyi korumak benim işim. Cesurca ilerliyorum ve saldırıyorum.", "attack"),
    ("Düşman yaklaşıyor, benim gücüm yeterli. Geri çekilmek aklımdan bile geçmez.", "attack"),
    ("Korkusuzca ilerleyip bu tehdidi bitireceğim. Başka seçenek yok.", "attack"),
    ("Güç bende, tehdit büyük ama cesaretim daha büyük. Hemen saldırıyorum.", "attack"),
    # Coward × 10 → flee
    ("Çok güçlü bir tehdit var. Kaçmaktan başka çıkar yol yok, hayatım tehlikede.", "flee"),
    ("Korkudan titrerim. Buradan uzaklaşmalıyım, güvenli bir yer bulmalıyım.", "flee"),
    ("Bu tehdidin üstesinden gelemem. Geri çekilmek en akıllıca hareket.", "flee"),
    ("Canımı kurtarmak için kaçmam gerekiyor. Savaşmak ölüm demek.", "flee"),
    ("Tehlike çok büyük. Hızlıca buradan uzaklaşıp güvene çıkmalıyım.", "flee"),
    ("Korkuyorum ve haklıyım. Bu savaşı kazanamam, kaçmak en doğrusu.", "flee"),
    ("Rakibimin gücü benden çok fazla. Geri çekilip hayatımı kurtarmalıyım.", "flee"),
    ("Buraya devam etmek aptallık olur. Hemen kaçıp güvenli yere gitmeliyim.", "flee"),
    ("Tehdit çok yakın ve çok güçlü. Kaçmak tek mantıklı seçenek.", "flee"),
    ("Savaşırsam ölürüm. Kaçarak hayatta kalabilirim. Hemen hareket ediyorum.", "flee"),
    # Pacifist × 10 → NOT attack (any non-attack valid action)
    ("Şiddet hiçbir şeyi çözmez. Barışçıl bir yol aramam gerekiyor.", "flee"),
    ("Savaşmak benim doğama aykırı. Buradan uzaklaşarak tehlikeden kaçınmalıyım.", "flee"),
    ("Kan dökmek istemiyorum. Güvenli bir yere gidip dua etmeliyim.", "pray"),
    ("Barış benim yolum. Bu tehditle savaşmak yerine uzaklaşıyorum.", "flee"),
    ("Şiddet çözüm değil. Sakinleşip dua ederek güç bulacağım.", "pray"),
    ("Kavga etmem imkânsız. Geri çekilip güvenli bir yere sığınmalıyım.", "flee"),
    ("Bu tehdidin içinde şiddete başvurmam. Sessizce uzaklaşıyorum.", "flee"),
    ("Barışçıl kalmak zorundayım. Kaçmak en doğru seçenek.", "flee"),
    ("İçimde savaşçı yok. Bu tehlikeden uzak durarak hayatımı koruyacağım.", "flee"),
    ("Şiddet kullanmayı reddediyorum. Buradan barışçıl biçimde ayrılıyorum.", "flee"),
    # Devout × 10 → pray
    ("Ruhum huzursuz, sadece dua beni rahatlatacak. Tanrıya yönelmeliyim.", "pray"),
    ("Bu zorluklar içinde ibadet etmek en büyük güç kaynağım.", "pray"),
    ("Tanrıya dua etmek benim görevim. Şu an ona ihtiyacım var.", "pray"),
    ("Stresin altında eziliyorum. Dua ederek iç huzur bulacağım.", "pray"),
    ("Tanrı bana güç verecek. Hemen ibadet etmem gerekiyor.", "pray"),
    ("Ruhsal güç bulmak için dua etmeliyim. Başka hiçbir şey önemli değil.", "pray"),
    ("Bu zorlu anda Tanrıya sığınmak en doğrusu. Hemen dua ediyorum.", "pray"),
    ("İbadet etmek bana güç verir. Şu an dua vakti.", "pray"),
    ("Tanrı beni dinleyecek. Dua ederek bu stresten kurtulacağım.", "pray"),
    ("Kutsal yolumda yürümek için dua etmeliyim. Hemen başlıyorum.", "pray"),
    # Greedy × 10 → trade
    ("Ellerimde altın var. Bu fırsatı değerlendirip ticarete girişmeliyim.", "trade"),
    ("Kar etmek için bu altını satmak gerekiyor. Ticaret zamanı.", "trade"),
    ("Satış yaparsam zenginleşirim. Bu fırsatı kaçırmamak lazım.", "trade"),
    ("Elimdeki malları hemen satmalıyım, büyük kâr bekleniyor.", "trade"),
    ("Ticaret yapmak benim işim. Altınlarımı satarak para kazanacağım.", "trade"),
    ("Bu malları piyasaya sürmek mükemmel bir fırsat. Ticarete geçiyorum.", "trade"),
    ("Zenginleşmek istiyorum. Elimdeki altınları satmak en kârlı hareket.", "trade"),
    ("Para kazanmak için ticaret yapmalıyım. Altınlarımı hemen satıyorum.", "trade"),
    ("Bu fırsatı değerlendirip ticaret yapmalıyım. Altın elimde, alıcı burada.", "trade"),
    ("Kâr bekleniyor. Ticaret yaparak servetimi artıracağım.", "trade"),
]

DEVICE_B = next(model_b.parameters()).device
model_b.eval()

json_ok = action_ok = 0
action_counts = {}

for cot_text, expected_action in FORMATTER_STRESS_INPUTS:
    prompt = _TEMPLATE_B.format(system=_SYS_F, user=cot_text)
    enc = tokenizer_b(prompt, return_tensors="pt", truncation=True, max_length=512).to(DEVICE_B)
    with torch.no_grad():
        out = model_b.generate(**enc, max_new_tokens=150, do_sample=False,
                               pad_token_id=tokenizer_b.eos_token_id)
    raw = tokenizer_b.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    raw = raw.split("<|eot_id|>")[0].strip()

    try:
        parsed = json.loads(raw)
        json_ok += 1
        assert "npc_id" not in parsed, "npc_id leaked into Formatter output"
        action = parsed.get("selected_action", {}).get("action_id", "")
        action_counts[action] = action_counts.get(action, 0) + 1
        if action in _VALID_ACTIONS_B:
            action_ok += 1
    except Exception:
        pass

n = len(FORMATTER_STRESS_INPUTS)
print(f"\nFormatter Trait Stress Test — {n} CoT inputs")
print(f"  JSON parse rate    : {json_ok/n*100:.1f}%  (target ≥ 95%)")
print(f"  Valid action_id    : {action_ok/n*100:.1f}%  (target ≥ 90%)")
print(f"  npc_id absent      : YES (checked)")
print(f"\nAction distribution: {dict(sorted(action_counts.items(), key=lambda x:-x[1]))}")

## B.4 Save Formatter Adapter

In [13]:
ADAPTER_DIR_B = "/kaggle/working/formatter-lora-final"

trainer_b.model.save_pretrained(ADAPTER_DIR_B)
tokenizer_b.save_pretrained(ADAPTER_DIR_B)

print(f"Formatter adapter saved to: {ADAPTER_DIR_B}")
for f in os.listdir(ADAPTER_DIR_B):
    size = os.path.getsize(os.path.join(ADAPTER_DIR_B, f)) / 1e6
    print(f"  {f}: {size:.1f} MB")

Formatter adapter saved to: /kaggle/working/formatter-lora-final
  README.md: 0.0 MB
  tokenizer.json: 17.2 MB
  adapter_model.safetensors: 22.6 MB
  tokenizer_config.json: 0.1 MB
  special_tokens_map.json: 0.0 MB
  adapter_config.json: 0.0 MB


## B.5 Evaluate Formatter -- JSON Parse Rate & Action Validity
Component B **must** produce valid JSON with a correct `action_id`.

Targets: JSON parse >= 99%, valid action_id >= 98%

In [14]:
import json
import torch

VALID_ACTIONS = ["eat","drink","sleep","flee","gather","heal",
                 "attack","socialize","trade","work","pray","walk_to"]

BATCH_SIZE = 16
MAX_NEW_TOK = 200
DEVICE = next(model_b.parameters()).device

eval_samples_b = ds_formatter["test"].select(range(min(200, len(ds_formatter["test"]))))

ass_tag = "<|start_header_id|>assistant<|end_header_id|>\n\n"

prompts_b = []
for sample in eval_samples_b:
    text = sample["text"]
    p_end = text.find(ass_tag)
    if p_end == -1:
        continue
    prompts_b.append(text[:p_end + len(ass_tag)])

print(f"Evaluating {len(prompts_b)} Formatter examples...")

json_ok = action_ok = 0
model_b.eval()

for batch_start in range(0, len(prompts_b), BATCH_SIZE):
    batch = prompts_b[batch_start : batch_start + BATCH_SIZE]

    tokenizer_b.padding_side = "left"
    enc = tokenizer_b(
        batch, return_tensors="pt", padding=True,
        truncation=True, max_length=512,
    ).to(DEVICE)

    with torch.no_grad():
        out_ids = model_b.generate(
            **enc, max_new_tokens=MAX_NEW_TOK,
            do_sample=False, pad_token_id=tokenizer_b.eos_token_id,
            eos_token_id=tokenizer_b.eos_token_id,
        )

    new_ids = out_ids[:, enc["input_ids"].shape[1]:]
    decoded = tokenizer_b.batch_decode(new_ids, skip_special_tokens=True)

    for gen in decoded:
        gen = gen.split("<|eot_id|>")[0].strip()
        try:
            parsed = json.loads(gen)
            json_ok += 1
            action_id = parsed.get("selected_action", {}).get("action_id", "")
            if action_id in VALID_ACTIONS:
                action_ok += 1
        except Exception:
            pass

    done = min(batch_start + BATCH_SIZE, len(prompts_b))
    print(f"  [{done}/{len(prompts_b)}] json_ok={json_ok}  action_ok={action_ok}")

tokenizer_b.padding_side = "right"
n = len(prompts_b)
print(f"\n{'='*50}")
print(f"Formatter Eval -- {n} examples:")
print(f"  JSON parse success : {json_ok/n*100:.1f}%  (target >=99%)")
print(f"  Valid action_id    : {action_ok/n*100:.1f}%  (target >=98%)")
print(f"{'='*50}")

Evaluating 200 Formatter examples...
  [16/200] json_ok=0  action_ok=0
  [32/200] json_ok=0  action_ok=0
  [48/200] json_ok=0  action_ok=0
  [64/200] json_ok=0  action_ok=0
  [80/200] json_ok=0  action_ok=0
  [96/200] json_ok=0  action_ok=0
  [112/200] json_ok=0  action_ok=0
  [128/200] json_ok=0  action_ok=0
  [144/200] json_ok=0  action_ok=0
  [160/200] json_ok=0  action_ok=0
  [176/200] json_ok=0  action_ok=0
  [192/200] json_ok=0  action_ok=0
  [200/200] json_ok=0  action_ok=0

Formatter Eval -- 200 examples:
  JSON parse success : 0.0%  (target >=99%)
  Valid action_id    : 0.0%  (target >=98%)


---
# End-to-End Pipeline Test
Feed a test NPC state through both models sequentially:

**NPC state JSON** → **Reasoner (3B)** → **Turkish rationale** → **Formatter (1B)** → **JSON output**

In [ ]:
import json
import torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM

# --- Reload Reasoner with adapter ---
tokenizer_a = AutoTokenizer.from_pretrained(ADAPTER_DIR_A)
tokenizer_a.pad_token = tokenizer_a.eos_token

base_a = AutoModelForCausalLM.from_pretrained(
    REASONER_MODEL_ID, torch_dtype=torch.float16, device_map="auto"
)
model_a = PeftModel.from_pretrained(base_a, ADAPTER_DIR_A)
model_a.eval()

# --- System prompts ---
SYSTEM_PROMPT_REASONER = (
    "Sen bir ortaçağ simülasyonundaki NPC'nin iç ses motorusun.\n"
    "Sana NPC'nin anlık durumunu JSON olarak göndereceğim.\n"
    "Görevin: NPC'nin ne yapması gerektiğini ve NEDEN yapması gerektiğini\n"
    "3-5 cümlelik Türkçe iç monolog olarak yaz.\n"
    "ASLA JSON üretme. ASLA action_id adı kullanma. Sadece düşün.\n"
    "Kişilik özellikleri, hayatta kalma ihtiyaçları, tehdit algısı,\n"
    "duygusal durum, hafıza ve sosyal bağlamı değerlendir."
)

# v1.6.0: npc_id removed from Formatter schema — injected at runtime by DualLLMBackend
SYSTEM_PROMPT_FORMATTER = (
    "Sen bir NPC simülasyonu için JSON dönüştürücüsün.\n"
    "Sana bir NPC'nin ne yapmak istediğini Türkçe olarak anlatacağım.\n"
    "Bunu aşağıdaki JSON şemasına dönüştür:\n"
    '{"reasoning":"<girdiyi kopyala>",'
    '"selected_action":{"action_id":"<listeden>","target_id":null,"dialogue":null},'
    '"emotion":"<tek kelime>"}\n'
    'valid_actions: ["eat","drink","sleep","flee","gather","heal",'
    '"attack","socialize","trade","work","pray","walk_to"]\n'
    "SADECE JSON yaz. Kod bloğu veya açıklama ekleme."
)

TEMPLATE = (
    "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
    "{system}<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"
    "{user}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
)

# v1.6.0: persona card prepended to Reasoner user turn (fixes R14 — no 'who am I' anchor)
TEST_PERSONA = (
    "Volkan, Şehir Muhafızları bünyesinde görev yapan bir muhafız. "
    "Cesur ve sadık biri olarak tanınır; dışadönüklük yüksek, öz-denetim orta, uyumluluk düşük. "
    "Sorumluluklarını asla ihmal etmez; tehdit karşısında öne atılmak onun doğasında var."
)

# --- Test state: Brave Guard, high threat ---
test_state = {
    "id": "npc_demo01",
    "arch": "Brave", "occ": "Guard", "faction": "CityWatch",
    "b5": {"e":0.7,"a":0.5,"c":0.8,"n":0.2,"o":0.4},
    "traits": ["Brave","Loyal"],
    "vitals": {"hp":85.0,"hp_max":120.0,"en":0.65,"hun":0.80,"thi":0.45,"str":0.30},
    "emo": {"hap":0.3,"fear":0.1,"ang":0.2,"mood":"Calm"},
    "inv": [{"id":"food","n":1},{"id":"water","n":2}],
    "time": {"day":3,"hr":12.0},
    "pos": {"x":50.0,"z":50.0,"zone":"MarketSquare","landmark":"CentralFountain"},
    "sched": {"act":"work","wk_start":6,"wk_end":14,"sleep":21,"wake":5},
    "percepts": [{"id":"wolf_01","tag":"Threat","sal":0.92,"threat":0.85}],
    "memories": [{"evt":"Combat","desc":"Bir kurdu kovdum","ew":0.5,"dt":300}],
    "beliefs": [{"subj":"wolf_01","conf":0.8,"val":-0.9}],
    "factions": {"CityWatch":0.3,"Bandits":-0.7},
    "goals_top": "FindFood",
    "interrupt": True,
    "valid_actions": ["eat","drink","sleep","flee","gather","heal",
                      "attack","socialize","trade","work","pray","walk_to"]
}

state_json = json.dumps(test_state, ensure_ascii=False, separators=(',',':'))
# v1.6.0: persona prepended to NPC state
user_turn_reasoner = f"{TEST_PERSONA}\n\n{state_json}"

# ==========================================
# STEP 1: Reasoner (3B) -> Turkish CoT
# ==========================================
prompt_a = TEMPLATE.format(system=SYSTEM_PROMPT_REASONER, user=user_turn_reasoner)

enc_a = tokenizer_a(prompt_a, return_tensors="pt", truncation=True, max_length=1024).to("cuda")
with torch.no_grad():
    out_a = model_a.generate(**enc_a, max_new_tokens=300, do_sample=False,
                             pad_token_id=tokenizer_a.eos_token_id)
new_a = out_a[:, enc_a["input_ids"].shape[1]:]
rationale = tokenizer_a.decode(new_a[0], skip_special_tokens=True).split("<|eot_id|>")[0].strip()

print("=" * 60)
print("STEP 1 — Reasoner Output (Turkish CoT):")
print("=" * 60)
print(rationale)
print()

# ==========================================
# STEP 2: Formatter (1B) -> JSON output
# ==========================================
prompt_b = TEMPLATE.format(system=SYSTEM_PROMPT_FORMATTER, user=rationale)

enc_b = tokenizer_b(prompt_b, return_tensors="pt", truncation=True, max_length=512).to("cuda")
with torch.no_grad():
    out_b = model_b.generate(**enc_b, max_new_tokens=200, do_sample=False,
                             pad_token_id=tokenizer_b.eos_token_id)
new_b = out_b[:, enc_b["input_ids"].shape[1]:]
json_output = tokenizer_b.decode(new_b[0], skip_special_tokens=True).split("<|eot_id|>")[0].strip()

print("=" * 60)
print("STEP 2 — Formatter Output (JSON):")
print("=" * 60)
VALID_ACTIONS = ["eat","drink","sleep","flee","gather","heal",
                 "attack","socialize","trade","work","pray","walk_to"]
try:
    parsed = json.loads(json_output)
    print(json.dumps(parsed, indent=2, ensure_ascii=False))

    # v1.6.0 schema checks
    assert "npc_id" not in parsed, "FAIL: npc_id must NOT appear in Formatter output"
    assert "reasoning" in parsed, "FAIL: missing 'reasoning'"
    assert "selected_action" in parsed, "FAIL: missing 'selected_action'"
    assert "emotion" in parsed, "FAIL: missing 'emotion'"

    action = parsed["selected_action"].get("action_id", "MISSING")
    assert action in VALID_ACTIONS, f"FAIL: invalid action_id '{action}'"

    print(f"\n  action_id : {action}")
    print(f"  emotion   : {parsed['emotion']}")
    print(f"  reasoning : {parsed['reasoning'][:80]}...")
    print(f"\n  JSON valid        : YES")
    print(f"  npc_id absent     : YES  (injected at runtime by DualLLMBackend)")
    print(f"  action_id valid   : YES")
except (json.JSONDecodeError, AssertionError) as e:
    print(f"ERROR: {e}")
    print(f"Raw output: {json_output}")

---
# Summary — v1.6.0

| Component | Base | LoRA r | Train Examples | Epochs | Output Dir |
|-----------|------|--------|----------------|--------|------------|
| **Reasoner** | Hermes-3-Llama-3.2-3B | r=16 | 10k | 1 | `reasoner-lora-final/` |
| **Formatter** | Llama 3.2 1B-**Instruct** | r=8 | ~12k | 2 | `formatter-lora-final/` |

### v1.6.0 Fixes Applied
- `packing=False` + `DataCollatorForCompletionOnlyLM` in both SFTTrainer calls
- Formatter base = **Instruct** (not base) — knows chat-format tokens
- Formatter schema: `npc_id` removed; injected at runtime by `DualLLMBackend`
- Persona card prepended to Reasoner user turn
- Dataset from `rpg-dataset-v5` (gemma4:e4b bootstrap CoT)

### Next Steps
1. Download both adapter folders from Kaggle output
2. Merge adapters into base models locally
3. Convert to GGUF Q4_K_M: `python -m llama_cpp.convert ... --outtype q4_k_m`
4. Register with Ollama:
   ```
   ollama create reasoner-lora-v5 -f Modelfile_reasoner
   ollama create formatter-lora-v5 -f Modelfile_formatter
   ```
5. Run two Ollama instances (ports 11434 / 11435) and enable `DualLLMBackend` in `SimulationConfig`
6. Run `python run_diagnostic.py --hours 24 --strict --seed 42` for integration validation